# Mitmproxy Implementation Challenges

Mitmproxy requires a custom CA to be installed as a system trust anchor, since user-installed certificates are rejected by apps unless explicitly opted in via their network security configuration. Installing a system CA requires root access, as the system certificate store resides in Android's read-only partitions. Additionally, Mitmproxy does not attribute captured traffic to specific apps; identifying connections by app UID requires supplementary iptables rules or eBPF kprobes. Finally, capturing QUIC traffic on Android requires Mitmproxy to operate in WireGuard mode rather than standard proxy mode, [since Mitmproxy does not support the QUIC protocol in proxy mode for Android](https://web.archive.org/web/20260820025124/https://github.com/mitmproxy/mitmproxy/issues/7025).

# Certificate Pinning Dataset Overview

Summary of the certificate pinning detection results and a table showing data recipients that perform certificate pinning, ranked by number of unique pinned hosts as reported by Mitmproxy.

**Paper reference:** $\S$ 6.3

While Mitmproxy does not provide a solution to Certificate Pinning, it can detect and log potential pinning attempts by analyzing the TLS handshake and logging them in the connection summary. When an app implements certificate pinning and detects an MITM attack, the client can either terminate the connection immediately or abruptly reset it. Sometimes, it can also send an explicit TLS Alert (Certificate Unknown) error. Additionally, an exception is raised and logged in the Logcat buffer, indicating that a certificate pinning check failed because the presented certificate or public key does not match the pinned certificate or public key. These exceptions include `SSLPeerUnverifiedException`, which is thrown when the client is unable to verify the peer's identity, and `CertificateException`, a generic certificate validation error is thrown when any part of the certificate validation process fails. 

Consequently, other exceptions, such as `SSLHandshakeException`, are also related to tracing the TLS handshake and checking whether the app uses pinning.
Among the 460 data recipients observed in our study, 64 perform certificate pinning, as indicated by Mitmproxy reporting connection failures during pin validation. Table shows the top data recipients by number of pinned hosts. ByteDance leads with 35 unique pinned hosts, followed by Google (26) and YouTube (17). Mitmproxy reported Meta performed pinning for 14 unique hosts, while Snapchat and CBS/Paramount reported 6 each. 

Furthermore, only 2 DRs, Button and Alaska Air, reported pinning failures with `java.security.cert.CertificateException`. Notably, pinning by ByteDance, Meta, Google, and Snapchat coincides with their use of custom TLS libraries, demonstrating the prevalence of instrumentation challenges across popular first- and third-party components.

In [2]:
# --- Cell: Pinning by Vantage Type ---
import pandas as pd
from analysis_utils.shared_setup import map_libs, mark_print
from tabulate import tabulate

http_df = pd.read_parquet("../data/http_requests.parquet")
pinning_df = pd.read_parquet("../data/pinning.parquet")
orig_pinning = pinning_df.copy()
advaita_df = pd.read_parquet("../data/observable_gaps.parquet")
advaita_df = map_libs(advaita_df)

# Pinning overview
mark_print(
    f"**Certificate pinning dataset:** "
    f"**{pinning_df['host'].nunique()}** unique hosts across "
    f"**{pinning_df['company'].nunique()}** data recipients."
)

pinning_by_vp = (
    pinning_df[["company", "vantage_type"]]
    .value_counts()
    .reset_index()
    .rename(columns={0: "count"})
    .groupby("vantage_type")
    .agg(hosts=("company", "count"), drs=("company", "nunique"))
    .reset_index()
    .rename(columns={"vantage_type": "Technique"})
)

mark_print("**Pinning detections per technique:**")
mark_print(tabulate(pinning_by_vp, headers="keys", tablefmt="pipe", showindex=False))

# --- Cell: Pinning and ByteDance case study ---
import pandas as pd
from analysis_utils.shared_setup import map_libs, mark_print
from tabulate import tabulate

http_df = pd.read_parquet("../data/http_requests.parquet")
pinning_df = pd.read_parquet("../data/pinning.parquet")
advaita_df = pd.read_parquet("../data/observable_gaps.parquet")
advaita_df = map_libs(advaita_df)

# Pinning overview
mark_print(
    f"**Certificate pinning dataset:** "
    f"**{pinning_df['host'].nunique()}** unique hosts across "
    f"**{pinning_df['company'].nunique()}** data recipients."
)

pinning_df = pinning_df[~pinning_df['app'].isna()]
pinning_df = pinning_df[~pinning_df['app'].str.contains('None', case=False, na=False)].reset_index(drop=True)

# Pinned hosts per DR
pinned_by_dr = (
    pinning_df.groupby("company")
    .agg(
        hosts=("host", "nunique"),
        host_list=("host", lambda x: ", ".join(sorted(x.unique()))),
        app_list=("app", lambda x: ", ".join(sorted(x.unique()))),
    )
    .sort_values("hosts", ascending=False)
    .reset_index()
    .rename(columns={"company": "Data Recipient", "hosts": "Pinned Hosts", 'app_list': 'Apps', "host_list": "Hosts"})
)

pinned_by_dr.drop(columns=["Hosts"], inplace=True)
mark_print("**Pinned hosts by data recipient:**")
mark_print(tabulate(pinned_by_dr, headers="keys", tablefmt="pipe", showindex=False))

**Certificate pinning dataset:** **292** unique hosts across **64** data recipients.

**Pinning detections per technique:**

| Technique   |   hosts |   drs |
|:------------|--------:|------:|
| Frida       |      62 |    62 |
| Mitm        |      11 |    11 |

**Certificate pinning dataset:** **292** unique hosts across **64** data recipients.

**Pinned hosts by data recipient:**

| Data Recipient            |   Pinned Hosts | Apps                                                                                                                                                                                                                                                                                                                                                      |
|:--------------------------|---------------:|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| ByteDance                 |             35 | com.android.location.fused, com.zhiliaoapp.musically                                                                                                                                                                                                                                                                                                      |
| Google                    |             25 | com.appmind.radios.gt, com.atpc, com.audiosdroid.arrangerkeyboard, com.diy.boba.drink, com.handmark.sportcaster, com.hypercarrot.giantrush, com.l, com.makeup.organizer.sorting.restock.life.simulation.asmr.game, com.noclip2.multiplayer, com.rejoy.block.logic, com.stockx.stockx, com.tpml.dh, qrcode.scanner.qrmaker, walkie.talkie.among.us.friends |
| YouTube                   |             17 | com.appmind.radios.gt, com.atpc, com.audiosdroid.arrangerkeyboard, com.pleasurecity.find.differences.spot.difference.puzzle                                                                                                                                                                                                                               |
| Meta                      |             14 | com.appmind.radios.gt, com.atpc, com.audiosdroid.arrangerkeyboard, com.doll.makeover.dressup.games, com.facebook.katana, com.hypercarrot.giantrush, com.instagram.android, com.instagram.barcelona, com.l, com.rejoy.block.logic, com.zhiliaoapp.musically                                                                                                |
| MobFox                    |             11 | com.appmind.radios.gt                                                                                                                                                                                                                                                                                                                                     |
| Snapchat                  |              6 | com.snapchat.android                                                                                                                                                                                                                                                                                                                                      |
| CBS/Paramount             |              6 | com.handmark.sportcaster                                                                                                                                                                                                                                                                                                                                  |
| Kuaishou                  |              5 | com.appmind.radios.gt, com.diy.boba.drink, com.hotplay.matchpuzzle, com.l, com.rejoy.block.logic, com.tpml.dh, walkie.talkie.among.us.friends                                                                                                                                                                                                             |
| InMobi                    |              5 | com.audiosdroid.arrangerkeyboard, com.hotplay.matchpuzzle, com.hypercarrot.giantrush, com.rejoy.block.logic, walkie.talkie.among.us.friends                                                                                                                                                                                                               |
| The Trade Desk            |              4 | com.hypercarrot.giantrush, com.rejoy.block.logic                                                                                                                                                                                                                                                                                                          |
| Google AdManager          |              4 | com.appmind.radios.gt, com.cookingcity.chef.kitchen.craze.fever, com.handmark.sportcaster, com.hotplay.matchpuzzle, com.kiloo.subwaysurf, com.rejoy.block.logic, com.stockx.stockx, walkie.talkie.among.us.friends                                                                                                                                        |
| ironSource                |              3 | com.ea.BejeweledBlitz_na, com.hypercarrot.giantrush, com.kiloo.subwaysurf, com.makeup.organizer.sorting.restock.life.simulation.asmr.game, com.noclip2.multiplayer, com.rejoy.block.logic, walkie.talkie.among.us.friends                                                                                                                                 |
| DoubleVerify              |              3 | com.hotplay.matchpuzzle, com.hypercarrot.giantrush, com.rejoy.block.logic, walkie.talkie.among.us.friends                                                                                                                                                                                                                                                 |
| Jampp                     |              3 | walkie.talkie.among.us.friends                                                                                                                                                                                                                                                                                                                            |
| Google AdMob              |              3 | com.appmind.radios.gt, com.diy.boba.drink, com.handmark.sportcaster, com.hypercarrot.giantrush, com.noclip2.multiplayer, com.quranmp3.readquran, com.rejoy.block.logic, com.stockx.stockx, hr.palamida, walkie.talkie.among.us.friends                                                                                                                    |
| Liftoff                   |              3 | com.hypercarrot.giantrush, com.makeup.organizer.sorting.restock.life.simulation.asmr.game, com.noclip2.multiplayer, com.rejoy.block.logic, net.difer.weather                                                                                                                                                                                              |
| Digital Turbine           |              3 | com.hotplay.matchpuzzle, com.rejoy.block.logic                                                                                                                                                                                                                                                                                                            |
| Magnite                   |              3 | com.rejoy.block.logic                                                                                                                                                                                                                                                                                                                                     |
| Uber                      |              3 | com.ubercab.eats                                                                                                                                                                                                                                                                                                                                          |
| Unity Technologies        |              3 | com.doll.makeover.dressup.games, com.hypercarrot.giantrush, com.makeup.organizer.sorting.restock.life.simulation.asmr.game                                                                                                                                                                                                                                |
| Pixalate                  |              2 | com.makeup.organizer.sorting.restock.life.simulation.asmr.game, com.rejoy.block.logic                                                                                                                                                                                                                                                                     |
| Moloco                    |              2 | com.audiosdroid.arrangerkeyboard, com.hotplay.matchpuzzle, com.hypercarrot.giantrush, com.rejoy.block.logic, qrcode.scanner.qrmaker                                                                                                                                                                                                                       |
| StackAdapt                |              2 | com.doll.makeover.dressup.games                                                                                                                                                                                                                                                                                                                           |
| Temu                      |              2 | com.atpc, com.rejoy.block.logic, walkie.talkie.among.us.friends                                                                                                                                                                                                                                                                                           |
| TopOn                     |              2 | com.hotplay.matchpuzzle, com.rejoy.block.logic                                                                                                                                                                                                                                                                                                            |
| Alaska Airlines           |              2 | com.alaskaairlines.android                                                                                                                                                                                                                                                                                                                                |
| Unknown                   |              2 | com.lelic.speedcam, com.makeup.organizer.sorting.restock.life.simulation.asmr.game, net.difer.weather                                                                                                                                                                                                                                                     |
| YSO Network               |              2 | com.noclip2.multiplayer                                                                                                                                                                                                                                                                                                                                   |
| PubMatic                  |              2 | com.doll.makeover.dressup.games, com.hotplay.matchpuzzle                                                                                                                                                                                                                                                                                                  |
| Mintegral                 |              2 | com.hotplay.matchpuzzle                                                                                                                                                                                                                                                                                                                                   |
| Index Exchange            |              2 | com.appmind.radios.gt, com.diy.boba.drink                                                                                                                                                                                                                                                                                                                 |
| Amazon                    |              2 | com.hypercarrot.giantrush                                                                                                                                                                                                                                                                                                                                 |
| AppsFlyer                 |              2 | com.hotplay.matchpuzzle, com.pleasurecity.find.differences.spot.difference.puzzle, com.zhiliaoapp.musically                                                                                                                                                                                                                                               |
| KPV                       |              1 | com.rejoy.block.logic                                                                                                                                                                                                                                                                                                                                     |
| ads_by_nimbus             |              1 | com.handmark.sportcaster                                                                                                                                                                                                                                                                                                                                  |
| Amazon Web Services (AWS) |              1 | com.hypercarrot.giantrush                                                                                                                                                                                                                                                                                                                                 |
| Xandr                     |              1 | com.diy.boba.drink                                                                                                                                                                                                                                                                                                                                        |
| Among Us Chat             |              1 | walkie.talkie.among.us.friends                                                                                                                                                                                                                                                                                                                            |
| AppLovin                  |              1 | com.hotplay.matchpuzzle                                                                                                                                                                                                                                                                                                                                   |
| TrustArc                  |              1 | com.hypercarrot.giantrush, com.rejoy.block.logic                                                                                                                                                                                                                                                                                                          |
| TradPlus                  |              1 | com.rejoy.block.logic                                                                                                                                                                                                                                                                                                                                     |
| Azure                     |              1 | com.rejoy.block.logic                                                                                                                                                                                                                                                                                                                                     |
| The Weather Network       |              1 | com.appmind.radios.gt, com.doll.makeover.dressup.games, com.hotplay.matchpuzzle                                                                                                                                                                                                                                                                           |
| BIGO                      |              1 | com.makeup.organizer.sorting.restock.life.simulation.asmr.game                                                                                                                                                                                                                                                                                            |
| Tenjin                    |              1 | com.rejoy.block.logic                                                                                                                                                                                                                                                                                                                                     |
| BeReal                    |              1 | com.bereal.ft                                                                                                                                                                                                                                                                                                                                             |
| Taboola                   |              1 | com.handmark.sportcaster                                                                                                                                                                                                                                                                                                                                  |
| BidMachine                |              1 | com.rejoy.block.logic                                                                                                                                                                                                                                                                                                                                     |
| Button                    |              1 | com.stockx.stockx                                                                                                                                                                                                                                                                                                                                         |
| Smadex                    |              1 | com.hypercarrot.giantrush                                                                                                                                                                                                                                                                                                                                 |
| SimWeather                |              1 | net.difer.weather                                                                                                                                                                                                                                                                                                                                         |
| RhythmOne                 |              1 | com.appmind.radios.gt                                                                                                                                                                                                                                                                                                                                     |
| QualMeas                  |              1 | com.lelic.speedcam                                                                                                                                                                                                                                                                                                                                        |
| Pubeasy                   |              1 | com.rejoy.block.logic                                                                                                                                                                                                                                                                                                                                     |
| Cluep                     |              1 | com.rejoy.block.logic                                                                                                                                                                                                                                                                                                                                     |
| Firebase                  |              1 | com.zhiliaoapp.musically                                                                                                                                                                                                                                                                                                                                  |
| Pangle by ByteDance       |              1 | com.rejoy.block.logic                                                                                                                                                                                                                                                                                                                                     |
| OpenJS Foundation         |              1 | com.ea.BejeweledBlitz_na, com.kiloo.subwaysurf, com.rejoy.block.logic, walkie.talkie.among.us.friends                                                                                                                                                                                                                                                     |
| Oojoovae                  |              1 | com.ubercab.eats                                                                                                                                                                                                                                                                                                                                          |
| Fit Analytics             |              1 | com.zzkko                                                                                                                                                                                                                                                                                                                                                 |
| Google Search             |              1 | com.appmind.radios.gt, com.audiosdroid.arrangerkeyboard                                                                                                                                                                                                                                                                                                   |
| Adjust                    |              1 | com.hotplay.matchpuzzle, com.rejoy.block.logic                                                                                                                                                                                                                                                                                                            |